# B2-023-generative-models-diffusion — Practice p15 — Solution

**Type:** proof · **Difficulty:** core · **Concepts:** denoising-diffusion-probabilistic-models

*45 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

## Solution

### (a) The claim and why independence belongs in it

$P(t)$: there is a random vector $\bar\varepsilon_t$ such that
$$x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\bar\varepsilon_t,\qquad\bar\varepsilon_t\sim N(0,I),$$
and $\bar\varepsilon_t$ is a function of $\varepsilon_1,\dots,\varepsilon_t$ only — hence independent of $x_0$ and of $\varepsilon_{t+1},\varepsilon_{t+2},\dots$.

The independence clause must be part of the hypothesis because the inductive step adds $\sqrt{\alpha_t}\sqrt{1-\bar\alpha_{t-1}}\,\bar\varepsilon_{t-1}$ to $\sqrt{\beta_t}\,\varepsilon_t$ and uses the sum rule, which requires the two Gaussian terms to be **independent**. Knowing only that $\bar\varepsilon_{t-1}\sim N(0,I)$ says nothing about its relation to $\varepsilon_t$; it is the "function of $\varepsilon_1,\dots,\varepsilon_{t-1}$ only" clause that guarantees independence from $\varepsilon_t$ (and from $x_0$, which the conditional statement $q(x_t\mid x_0)$ needs). Without it the induction cannot advance.

### (b) Base case and the $t=0$ padding

$t=1$: $x_1=\sqrt{\alpha_1}\,x_0+\sqrt{\beta_1}\,\varepsilon_1$. Since $\bar\alpha_1=\alpha_1$ and $1-\bar\alpha_1=\beta_1$, this is $\sqrt{\bar\alpha_1}x_0+\sqrt{1-\bar\alpha_1}\,\bar\varepsilon_1$ with $\bar\varepsilon_1=\varepsilon_1\sim N(0,I)$, a function of $\varepsilon_1$ alone and independent of $x_0,\varepsilon_2,\dots$ by assumption. So $P(1)$ holds.

$t=0$: with $\bar\alpha_0=1$ (the empty product), the formula reads $x_0=1\cdot x_0+0\cdot\bar\varepsilon_0$, which is true whatever $\bar\varepsilon_0$ is, because its coefficient $\sqrt{1-\bar\alpha_0}$ is $0$. So the padding makes the closed form (and $q(x_0\mid x_0)$ as a point mass) consistent at $t=0$. The independence clause is vacuous there (no noise has been drawn), which is why the induction is anchored at $t=1$.

### (c) Inductive step $P(t-1)\Rightarrow P(t)$, $t\ge2$

By the forward step and $P(t-1)$,
$$x_t=\sqrt{\alpha_t}\,x_{t-1}+\sqrt{\beta_t}\,\varepsilon_t=\sqrt{\alpha_t\bar\alpha_{t-1}}\,x_0+\underbrace{\sqrt{\alpha_t(1-\bar\alpha_{t-1})}\,\bar\varepsilon_{t-1}}_{U}+\underbrace{\sqrt{\beta_t}\,\varepsilon_t}_{W}.$$
First, $\alpha_t\bar\alpha_{t-1}=\bar\alpha_t$ by the definition of the cumulative product.
**The two Gaussian terms.** $U\sim N(0,\alpha_t(1-\bar\alpha_{t-1})I)$ by Book 1's $\operatorname{Var}(cX)=c^2\operatorname{Var}X$, and $W\sim N(0,\beta_tI)$ likewise.
**Independence.** $U$ is a function of $\varepsilon_1,\dots,\varepsilon_{t-1}$ (by $P(t-1)$) and $W$ of $\varepsilon_t$; the $\varepsilon$'s are mutually independent, so $U$ and $W$ are independent.
**Sum rule.** $U+W\sim N\big(0,(\alpha_t(1-\bar\alpha_{t-1})+\beta_t)I\big)$, and
$$\alpha_t(1-\bar\alpha_{t-1})+\beta_t=\alpha_t-\alpha_t\bar\alpha_{t-1}+(1-\alpha_t)=1-\bar\alpha_t .$$
Define $\bar\varepsilon_t=(U+W)/\sqrt{1-\bar\alpha_t}$ (note $1-\bar\alpha_t>0$ since each $0<\alpha_s<1$). Then $\bar\varepsilon_t\sim N(0,I)$, it is a function of $\varepsilon_1,\dots,\varepsilon_t$ only — so it is independent of $x_0$ and of $\varepsilon_{t+1},\dots$ — and $x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\bar\varepsilon_t$. This is $P(t)$.

### (d) Conclusion and the reused-noise defect

By induction $P(t)$ holds for all $t\ge1$. Conditioning on $x_0$ (which is independent of $\bar\varepsilon_t$) leaves $\sqrt{\bar\alpha_t}x_0$ as a constant shift of a $N(0,(1-\bar\alpha_t)I)$ vector:
$$q(x_t\mid x_0)=N\big(\sqrt{\bar\alpha_t}\,x_0,\ (1-\bar\alpha_t)I\big).$$

**If every step reused the same draw** ($\varepsilon_t=\varepsilon_1$), the step in (c) breaks at independence: $U$ and $W$ are both multiples of the same $\varepsilon_1$, perfectly correlated, so the sum rule does not apply and the covariance term $2\operatorname{Cov}(U,W)$ survives. At $t=2$,
$$x_2=\sqrt{\bar\alpha_2}\,x_0+\big(\sqrt{\alpha_2\beta_1}+\sqrt{\beta_2}\big)\varepsilon_1,$$
whose noise variance is
$$\big(\sqrt{\alpha_2\beta_1}+\sqrt{\beta_2}\big)^2=(1-\beta_2)\beta_1+\beta_2+2\sqrt{(1-\beta_2)\beta_1\beta_2}=1-\bar\alpha_2+2\sqrt{(1-\beta_2)\beta_1\beta_2},$$
too large by the cross term $2\sqrt{(1-\beta_2)\beta_1\beta_2}$.

### (e) Numbers for $\beta=(0.1,0.3)$

$\alpha=(0.9,0.7)$ and $\bar\alpha_2=0.63$.
*Composing the steps:* noise variance $\alpha_2\beta_1+\beta_2=0.7\cdot0.1+0.3=0.37$.
*Closed form:* $1-\bar\alpha_2=1-0.63=0.37$. They agree.
(With reused noise it would be $0.37+2\sqrt{0.7\cdot0.1\cdot0.3}=0.37+0.289828=0.659828$.)

In [ ]:
import math

import torch

beta = (0.1, 0.3)
alpha = tuple(1 - b for b in beta)
alpha_bar_2 = alpha[0] * alpha[1]
var_composed = alpha[1] * beta[0] + beta[1]
var_closed = 1 - alpha_bar_2
var_reused = (math.sqrt(alpha[1] * beta[0]) + math.sqrt(beta[1])) ** 2

# Monte Carlo: run the two steps with independent and with reused noise
g = torch.Generator().manual_seed(20261015)
n = 400_000
x0 = torch.full((n,), 1.5, dtype=torch.float64)
e1 = torch.randn(n, generator=g, dtype=torch.float64)
e2 = torch.randn(n, generator=g, dtype=torch.float64)
x1 = math.sqrt(alpha[0]) * x0 + math.sqrt(beta[0]) * e1
x2_indep = math.sqrt(alpha[1]) * x1 + math.sqrt(beta[1]) * e2
x2_reused = math.sqrt(alpha[1]) * x1 + math.sqrt(beta[1]) * e1
print(f"alpha_bar_2 {alpha_bar_2:.6f}  composed {var_composed:.6f}  closed {var_closed:.6f}  reused {var_reused:.6f}")
print(f"MC independent: mean {x2_indep.mean().item():.4f} var {x2_indep.var().item():.4f};  reused var {x2_reused.var().item():.4f}")

### Answer check

In [ ]:
assert math.isclose(alpha_bar_2, 0.63, abs_tol=1e-12, rel_tol=1e-09)
assert math.isclose(var_composed, 0.37, abs_tol=1e-12, rel_tol=1e-09) and math.isclose(var_closed, 0.37, abs_tol=1e-12, rel_tol=1e-09)
assert math.isclose(var_reused, var_closed + 2 * math.sqrt(alpha[1] * beta[0] * beta[1]), abs_tol=1e-12, rel_tol=1e-09)
assert abs(x2_indep.mean().item() - math.sqrt(alpha_bar_2) * 1.5) < 0.01
assert abs(x2_indep.var().item() - var_closed) < 0.01
assert abs(x2_reused.var().item() - var_reused) < 0.01